# Module 2: Mixed Complementarity Problems (MCPs) and Strategic Competition

In this module we inspect market-equilibrium problems expressed as **mixed complementarity problems (MCPs)**. The modeling exercises in this notebook include:

1)  Duopoly energy producer problem

2)  Energy network market equilibrium

#### Activate the course environment and import packages

In [15]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using JuMP, PATHSolver


  Activating project at `~/Desktop/git/Equilibrium_modeling_course_Berlin2026`


## Exercise 2.1: Duopoly competition model

#### Problem Description

We analyze a **duopoly competition model** where two energy producers compete in quantities (Cournot competition). The problem is formulated as a **Linear Complementarity Problem (LCP)** derived from the first-order conditions of profit maximization.

#### Mathematical Formulation

**Complementarity Conditions:**

$0 \leq q_i \perp F(q_i) \geq 0$ for $i = 1,2$

where:

$F\begin{pmatrix} q_1 \\ q_2 \end{pmatrix} = \begin{pmatrix} 2\beta q_1 + \beta q_2 - \alpha + \gamma_1 \\ \beta q_1 + 2\beta q_2 - \alpha + \gamma_2 \end{pmatrix}$


**Parameters:**
- $\alpha = 10$ (market demand intercept)
- $\beta = 5$ (demand slope parameter)
- $\gamma_1 = 1$ (producer 1's cost parameter)
- $\gamma_2 = 1$ (producer 2's cost parameter)

Notice that this problem consists of **only complementarity conditions**, without an objective function. Hence, we need to use the **PATH** solver to solve it.

#### Define Problem Data

In [16]:
# Cost Parameter
gamma1 = 1
gamma2 = 1

# Demand function parameters
a = 10
b = 5;

### Task 1: Implementation

Construct and solve the model. For convenience, notice that we implement $F(q_i)$ as a single expression instead of $F(q_1)$ and $F(q_2)$ separately. $$F(q_i) = \beta q_i + \beta \sum_{j \in I} q_j - \alpha + \gamma_i.$$

In JuMP, a complementarity condition $0 \leq F(x) \perp x \geq 0$ is written with the `⟂` operator (type `\perp<TAB>`) inside `@constraint`, pairing the expression with the nonnegative variable it complements.

In [17]:
function duopoly_problem(gamma1, gamma2)

    # Initialize duopoly model (m)
    m = Model(PATHSolver.Optimizer)
    set_silent(m)

    # Set of firms
    I = [1, 2]

    # Define parameters
    gamma = Dict(1 => gamma1, 2 => gamma2)

    # Define variables
    @variable(m, q[I] >= 0)

    # Complementarity constraints
    # 0 ≤ q_i ⟂ (gamma_i - a + b*sum(q_j)) ≥ 0
    @constraint(m, [i in I], b*q[i] + b*sum(q[j] for j in I) - a + gamma[i] ⟂ q[i])

    # Solve the model using the PATH solver
    optimize!(m)

    # Print termination status
    println("\nSolver Status: ", termination_status(m))

    # Return model object with results
    return m
end

duopoly_problem (generic function with 1 method)

Function for displaying results

In [18]:
function display_duopoly_results(m, gamma1, gamma2)
    # Display results
    println("\n", "="^60)
    println("DUOPOLY EQUILIBRIUM RESULTS")
    println("="^60)
    println("Marginal Costs: γ₁ = ", gamma1, ", γ₂ = ", gamma2)
    println("Demand: P = ", a, " - ", b, "*(q₁ + q₂)")
    println("\nEquilibrium Quantities:")

    q1_val = value(m[:q][1])
    q2_val = value(m[:q][2])
    Q_total = q1_val + q2_val
    P_market = a - b * Q_total

    println("  Firm 1: q₁ = ", round(q1_val, digits=2))
    println("  Firm 2: q₂ = ", round(q2_val, digits=2))
    println("  Total: Q = ", round(Q_total, digits=2))

    println("\nMarket Price: P = ", round(P_market, digits=2))

    # Calculate profits
    profit1 = (P_market - gamma1) * q1_val
    profit2 = (P_market - gamma2) * q2_val

    println("\nProfits:")
    println("  Firm 1: π₁ = ", round(profit1, digits=2))
    println("  Firm 2: π₂ = ", round(profit2, digits=2))
    println("  Total: Π = ", round(profit1 + profit2, digits=2))
    println("="^60, "\n")
end

display_duopoly_results (generic function with 1 method)

Solve the model with given data

In [19]:
duopoly_model = duopoly_problem(gamma1, gamma2)
display_duopoly_results(duopoly_model, gamma1, gamma2)


Solver Status: LOCALLY_SOLVED

DUOPOLY EQUILIBRIUM RESULTS
Marginal Costs: γ₁ = 1, γ₂ = 1
Demand: P = 10 - 5*(q₁ + q₂)

Equilibrium Quantities:
  Firm 1: q₁ = 0.6
  Firm 2: q₂ = 0.6
  Total: Q = 1.2

Market Price: P = 4.0

Profits:
  Firm 1: π₁ = 1.8
  Firm 2: π₂ = 1.8
  Total: Π = 3.6



### Task 2: Sensitivity Analysis

Now suppose you are energy producer 1 and want to drive your competitor (producer 2) out of business. You can only control your costs $\gamma_1$ but could make them go negative (e.g., get subsidized by the government).

Do an analysis (numerical and/or analytical) to see the largest value of $\gamma_1$ that forces $q_2=0$ (put your competitor out of business).

In [20]:
# Change these values to see how the results change
gamma1_new = 1

duopoly_new = duopoly_problem(gamma1_new, gamma2)
display_duopoly_results(duopoly_new, gamma1_new, gamma2)


Solver Status: LOCALLY_SOLVED

DUOPOLY EQUILIBRIUM RESULTS
Marginal Costs: γ₁ = 1, γ₂ = 1
Demand: P = 10 - 5*(q₁ + q₂)

Equilibrium Quantities:
  Firm 1: q₁ = 0.6
  Firm 2: q₂ = 0.6
  Total: Q = 1.2

Market Price: P = 4.0

Profits:
  Firm 1: π₁ = 1.8
  Firm 2: π₂ = 1.8
  Total: Π = 3.6



### Task 3: Adding capacity constraints

With capacity constraint $q_i \leq q_i^{\max}$, producer 1's problem becomes:

\begin{align*}
\text{Maximize}_{q_1} \quad & (\alpha - \beta (q_1 + q_2)) q_1 - \gamma_1 q_1 \\
\text{s.t.} \quad & q_1 - q_1^{\max} \leq 0 \qquad (\delta_1) \\
& - q_1 \leq 0
\end{align*}

1) Derive the KKT conditions for this problem and reformulate the duopoly model as an MCP. Implement and solve the new model with $q_1^{\max} = 0.5$ and $q_2^{\max} = 2$.

2) Do the same sensitivity analysis as in Task 2 and find values for company 1 that force the competitor out of business.

In [21]:
function duopoly_problem_with_capacity_constraints(gamma1, gamma2, q_max1, q_max2)

    # Initialize duopoly model (m)
    m = Model(PATHSolver.Optimizer)
    set_silent(m)

    # Set of firms
    I = [1, 2]

    # Define parameters
    gamma = Dict(1 => gamma1, 2 => gamma2)
    q_max = Dict(1 => q_max1, 2 => q_max2)

    # Define variables
    @variable(m, q[I] >= 0)
    @variable(m, delta[I] >= 0)

    # Complementarity constraints
    @constraint(m, [i in I], b*q[i] + b*sum(q[j] for j in I) - a + gamma[i] + delta[i] ⟂ q[i])
    @constraint(m, [i in I], q_max[i] - q[i] ⟂ delta[i])

    # Solve the model using the PATH solver
    optimize!(m)

    # Print termination status
    println("\nSolver Status: ", termination_status(m))

    # Return model object with results
    return m
end

duopoly_problem_with_capacity_constraints (generic function with 1 method)

Repeat the same analysis as above and find values for company 1 that force the competitor out of business. This time you can also control the capacity $q_1^{\max}$.

In [22]:
# Change these values to see how the results change
gamma1_new = 1
q1_max = 0.5

# Do not change this value
q2_max = 1

duopoly_model_with_capacities = duopoly_problem_with_capacity_constraints(gamma1_new, gamma2, q1_max, q2_max)
display_duopoly_results(duopoly_model_with_capacities, gamma1_new, gamma2)


Solver Status: LOCALLY_SOLVED

DUOPOLY EQUILIBRIUM RESULTS
Marginal Costs: γ₁ = 1, γ₂ = 1
Demand: P = 10 - 5*(q₁ + q₂)

Equilibrium Quantities:
  Firm 1: q₁ = 0.5
  Firm 2: q₂ = 0.65
  Total: Q = 1.15

Market Price: P = 4.25

Profits:
  Firm 1: π₁ = 1.62
  Firm 2: π₂ = 2.11
  Total: Π = 3.74



## Exercise 2.2: Spatial Price Equilibrium



### Problem Description

We analyze a network with four supply and 5 demand nodes, i.e. with a total of 20 transportation arcs. 
Recall from the lecture: 
$$
c_{ij} + \psi_i(S_i) - \theta_j(D_j) \ge 0,
\qquad
x_{ij} \ge 0,
\qquad
i=I,\; j=J
$$

$$
S_i = \sum_{j=1}^{3} x_{ij},
\qquad i=I
$$

$$
D_j = \sum_{i=1}^{2} x_{ij},
\qquad j=J
$$

Based on 4.4.3.1, let us assume that inverse demand and supply follow a linear function: 

$$
\psi_i(x)
=
\kappa_i^S
+
\rho_i^S
\left(
\sum_j x_{ij}
\right)
$$

$$
\theta_j(x)
=
\kappa_j^D
-
\rho_j^D
\left(
\sum_i x_{ij}
\right)
$$

with the following supply, cost and demand parameters: 

### Supply parameters

| $i$ | $\kappa_i^S$ | $\rho_i^S$ |
|---:|---:|---:|
| 1 | $0$ | $\frac{1}{20}$ |
| 2 | $0$ | $\frac{1}{5}$ |
| 3 | $-7$ | $\frac{1}{5}$ |
| 4 | $0$ | $\frac{1}{5}$ |

### Demand parameters

| $j$ | $\kappa_j^D$ | $\rho_j^D$ |
|---:|---:|---:|
| 1 | $29$ | $1$ |
| 2 | $46$ | $1$ |
| 3 | $15$ | $1$ |
| 4 | $13$ | $1$ |
| 5 | $42$ | $1$ |

### Cost parameters

| From \ To | 1 | 2 | 3 | 4 | 5 |
|---:|---:|---:|---:|---:|---:|
| 1 | 10 | 20 | 30 | 5 | 6 |
| 2 | 3 | 15 | 12 | 5 | 6 |
| 3 | 17 | 40 | 5 | 3 | 7 |
| 4 | 50 | 23 | 28 | 12 | 4 |

### Task 2.2.1: Basic Model

Implement the setting as a complementarity model and solve for the resulting equilibrium. 

In [23]:
function build_4_4_3_1_complementarity_model(I, J, κS, ρS, κD, ρD)
    model = Model(PATHSolver.Optimizer)

    @variable(model, x[I, J] >= 0)

    # Supply and demand quantities:
    @expression(model, S[i in I], sum(x[i,j] for j in J))
    @expression(model, D[j in J], sum(x[i,j] for i in I))

    # Supply prices:
    @expression(
        model,
        ψ[i in I],
        κS[i] + ρS[i] * S[i]
    )

    # Demand prices:
    @expression(
        model,
        θ[j in J],
        κD[j] - ρD[j] * D[j]
    )

    # equilibrium condition
    @constraint(
        model,
        equilibrium[i in I, j in J],
            c[i,j] + ψ[i] - θ[j] ⟂ x[i,j]
    )

    set_silent(model)
    optimize!(model)
    return model
end

function show_results(model, I, J)

    println("Termination status: ", termination_status(model))

    x_opt = value(model[:x])
    @show x_opt

    println("\nSupply:")
    for i in I
        println(
            "S[$i] = ", value(model[:S][i]),
            ", ψ[$i] = ", value(model[:ψ][i])
        )
    end

    println("\nDemand:")
    for j in J
        println(
            "D[$j] = ", value(model[:D][j]),
            ", θ[$j] = ", value(model[:θ][j])
        )
    end
    return nothing
end

show_results (generic function with 1 method)

In [24]:
I = 1:4
J = 1:5
κS = JuMP.Containers.DenseAxisArray(
    [0.0, 0.0, -7.0, 0.0],
    I
)
ρS = JuMP.Containers.DenseAxisArray(
    [1/20, 1/5, 1/5, 1/5],
    I
)
κD = JuMP.Containers.DenseAxisArray(
    [29.0, 46.0, 15.0, 13.0, 42.0],
    J
)
ρD = JuMP.Containers.DenseAxisArray(
    [1.0, 1.0, 1.0, 1.0, 1.0],
    J
)
c = JuMP.Containers.DenseAxisArray(
    [
        10.0  20.0  30.0   5.0   6.0
         3.0  15.0  12.0   5.0   6.0
        17.0  40.0   5.0   3.0   7.0
        50.0  23.0  28.0  12.0   4.0
    ],
    I, J
)

model = build_4_4_3_1_complementarity_model(I, J, κS, ρS, κD, ρD)

show_results(model, I, J)

Termination status: LOCALLY_SOLVED
x_opt = 2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:4
    Dimension 2, 1:5
And data, a 4×5 Matrix{Float64}:
  0.0              14.999994035049575   0.0                 0.0               5.00000867749403
 19.9999990411316  10.000005432436668   0.0                 0.0               0.0
  0.0               0.0                10.000000779733469  10.0000007798807  14.999994780736497
  0.0               0.0                 0.0                 0.0              14.999996952488694

Supply:
S[1] = 20.000002712543605, ψ[1] = 1.0000001356271802
S[2] = 30.000004473568268, ψ[2] = 6.000000894713653
S[3] = 34.99999634035066, ψ[3] = -7.31929866493175e-7
S[4] = 14.999996952488694, ψ[4] = 2.999999390497739

Demand:
D[1] = 19.9999990411316, θ[1] = 9.000000958868402
D[2] = 24.99999946748624, θ[2] = 21.00000053251376
D[3] = 10.000000779733469, θ[3] = 4.999999220266531
D[4] = 10.0000007798807, θ[4] = 2.9999992201192995
D[5] = 35.000000410

### Task 2.2.2: Sensitivity Analysis

Now, modify $ρ^S_1$ to be the same as all other parameters. 

In [25]:
I = 1:4
J = 1:5
κS = JuMP.Containers.DenseAxisArray(
    [0.0, 0.0, -7.0, 0.0],
    I
)
ρS = JuMP.Containers.DenseAxisArray(
    [1/5, 1/5, 1/5, 1/5],
    I
)
κD = JuMP.Containers.DenseAxisArray(
    [29.0, 46.0, 15.0, 13.0, 42.0],
    J
)
ρD = JuMP.Containers.DenseAxisArray(
    [1.0, 1.0, 1.0, 1.0, 1.0],
    J
)
c = JuMP.Containers.DenseAxisArray(
    [
        10.0  20.0  30.0   5.0   6.0
         3.0  15.0  12.0   5.0   6.0
        17.0  40.0   5.0   3.0   7.0
        50.0  23.0  28.0  12.0   4.0
    ],
    I, J
)

model = build_4_4_3_1_complementarity_model(I, J, κS, ρS, κD, ρD)

show_results(model, I, J)

Termination status: LOCALLY_SOLVED
x_opt = 2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:4
    Dimension 2, 1:5
And data, a 4×5 Matrix{Float64}:
  0.0                9.166666666648238  0.0                0.0                0.0
 19.16666666666295  15.000000000018797  0.0                0.0                0.0
  0.0                0.0                9.615384615384556  9.61538461538465  17.692307692307832
  0.0                0.0                0.0                0.0               16.923076923076774

Supply:
S[1] = 9.166666666648238, ψ[1] = 1.8333333333296478
S[2] = 34.16666666668175, ψ[2] = 6.83333333333635
S[3] = 36.92307692307703, ψ[3] = 0.3846153846154081
S[4] = 16.923076923076774, ψ[4] = 3.384615384615355

Demand:
D[1] = 19.16666666666295, θ[1] = 9.833333333337048
D[2] = 24.166666666667034, θ[2] = 21.833333333332963
D[3] = 9.615384615384556, θ[3] = 5.384615384615444
D[4] = 9.61538461538465, θ[4] = 3.3846153846153495
D[5] = 34.615384615384606, θ[5] = 7

## Exercise 2.3: Multi-Node Energy Network Equilibrium

### Problem Description

We analyze a **two-node energy network** where four producers compete as price-takers in an asymmetric market structure. This creates a complex equilibrium with production, trading, and network flow decisions.

### Network Structure:
- **Node 1**: Producers A and B (can produce locally and export to Node 2)
- **Node 2**: Producers C and D (can only produce locally)
- **Network Operator**: Manages transmission flow $g_{12}$ between nodes
- **Asymmetric Market**: Only Node 1 producers can participate in inter-nodal trade

##### Define Problem Data and Parameters

In [26]:
# Demand function parameters: D(π) = a - b*π
demand_params = Dict(
    "node1" => Dict("a" => 20, "b" => 1),  # D₁(π₁) = 20 - π₁
    "node2" => Dict("a" => 40, "b" => 2)   # D₂(π₂) = 40 - 2*π₂
)

# Production cost parameters: γᵢ (marginal cost)
production_costs = Dict("A" => 10, "B" => 12, "C" => 15, "D" => 18)

# Production capacity limits: q̄ᵢ
production_capacity = Dict("A" => 10, "B" => 10, "C" => 5, "D" => 5)

# Transmission parameters
transmission_params = Dict(
    "reg_tariff" => 0.5,  # τ₁₂ᴿᵉᵍ: Regulated transmission tariff
    "tso_cost" => 1,      # γᵀˢᴼ: TSO operational cost
    "capacity" => 5,      # ḡ₁₂: Maximum transmission capacity
)

# Print problem parameters
println("=== MULTI-NODE ENERGY NETWORK DATA ===")

println("\nDemand Functions:")
for node in ["node1", "node2"]
    params = demand_params[node]
    println("  ", node, ": D = ", params["a"], " - ", params["b"], "*π")
end

println("\nProduction Costs (\$/MWh):")
for producer in ["A", "B", "C", "D"]
    cost = production_costs[producer]
    node = producer in ("A", "B") ? 1 : 2
    capacity = production_capacity[producer]
    println("  Producer ", producer, " (Node ", node, "): \$", cost, "/MWh, Capacity: ", capacity, " MW")
end

println("\nTransmission Parameters:")
println("  Regulated tariff: \$", transmission_params["reg_tariff"], "/MWh")
println("  TSO cost: \$", transmission_params["tso_cost"], "/MWh")
println("  Transmission capacity: ", transmission_params["capacity"], " MW")

=== MULTI-NODE ENERGY NETWORK DATA ===

Demand Functions:
  node1: D = 20 - 1*π
  node2: D = 40 - 2*π

Production Costs ($/MWh):
  Producer A (Node 1): $10/MWh, Capacity: 10 MW
  Producer B (Node 1): $12/MWh, Capacity: 10 MW
  Producer C (Node 2): $15/MWh, Capacity: 5 MW
  Producer D (Node 2): $18/MWh, Capacity: 5 MW

Transmission Parameters:
  Regulated tariff: $0.5/MWh
  TSO cost: $1/MWh
  Transmission capacity: 5 MW


### Task 1: Implementation

##### Mathematical Formulation

We will implement this model in JuMP in parts and the formulation is shown in parts below. The formulation has

**Sets:**

{A, B} = Producers at Node 1

{C, D} = Producers at Node 2

{node1, node2} = Nodes in the network


**Parameters:**

$D_i(\pi_i)$ = demand function at node $i$

$\gamma_j$ = operational cost of producer $j$

$\bar{q}_i^j$ = production capacity limit for producer $j$ at node $i$

$\tau_{12}^{Reg}$ = regulated transmission tariff

$\gamma^{TSO}$ = TSO operational cost

$\bar{g}_{12}$ = maximum transmission capacity from node 1 to node 2

In [27]:
producers_node1 = ["A", "B"]
producers_node2 = ["C", "D"]
producers = vcat(producers_node1, producers_node2)
nodes = ["node1", "node2"]

# Demand function parameters: D(π) = a - b*π
demand_a = Dict("node1" => 20, "node2" => 40)
demand_b = Dict("node1" => 1, "node2" => 2)

# Production cost and capacity
γ = Dict("A" => 10, "B" => 12, "C" => 15, "D" => 18)
q_bar = Dict("A" => 10, "B" => 10, "C" => 5, "D" => 5)

# Transmission parameters
τ_reg = 0.5      # Regulated transmission tariff
γ_tso = 1;       # TSO operational cost

**Decision Variables:**

$s_i^j$ = local sales by producer $j$ at node $i$ \
$q_i^j$ = total production by producer $j$ at node $i$ \
$f_{12}^j$ = inter-nodal flow from node 1 to node 2 by producer $j$ \
$g_{12}$ = total transmission flow from node 1 to node 2 

**Dual variables:**

$\pi_i$ = energy price at node $i$ \
$\tau_{12}$ = endogenous transmission tariff \
$\delta_i^j$ = shadow cost of production balance constraint for producer $j$ at node $i$ \
$\lambda_i^j$ = shadow cost of capacity constraint for producer $j$ at node $i$ \
$\varepsilon_{12}$ = shadow cost of transmission capacity constraint

In [28]:
function add_variables(model)
    # Production and sales
    @variable(model, q[producers] >= 0)          # Production
    @variable(model, s[producers] >= 0)          # Local sales
    @variable(model, f[producers_node1] >= 0)    # Inter-nodal flow (A, B only)
    @variable(model, λ[producers] >= 0) # Capacity shadow price
    @variable(model, δ[producers])           # Production balance shadow price (free)

    # Market variables
    @variable(model, π_var[nodes])   # Nodal prices (free)
    @variable(model, τ)          # Endogenous transmission tariff (free)

    # Network operator's decision variables
    @variable(model, g12 >= 0)  # Transmission flow
    @variable(model, ϵ >= 0)  # Transmission capacity shadow price
end

add_variables (generic function with 1 method)

**Producer A (Node 1):**

$0 \leq -\pi_1 + \delta_1^A \perp s_1^A \geq 0$

$0 \leq \gamma_1^A + \lambda_1^A - \delta_1^A \perp q_1^A \geq 0$

$0 \leq -\pi_2 + (\tau_{12}^{Reg} + \tau_{12}) + \delta_1^A \perp f_{12}^A \geq 0$

$0 \leq \bar{q}_1^A - q_1^A \perp \lambda_1^A \geq 0$

$0 = s_1^A - q_1^A + f_{12}^A, \delta_1^A \text{ free}$

In [29]:
function add_node1_producer_constraints(m, p)
    # Pick up variables from the model
    q, s, f, λ, δ, π_var, τ = m[:q], m[:s], m[:f], m[:λ], m[:δ], m[:π_var], m[:τ]

    # Producer p (Node 1): s, q, f, λ complementarity conditions
    @constraint(m, -π_var["node1"] + δ[p] ⟂ s[p])
    @constraint(m, γ[p] + λ[p] - δ[p] ⟂ q[p])
    @constraint(m, -π_var["node2"] + (τ_reg + τ) + δ[p] ⟂ f[p])
    @constraint(m, q_bar[p] - q[p] ⟂ λ[p])

    # Production balance (equality; δ[p] is the free multiplier)
    @constraint(m, s[p] - q[p] + f[p] ⟂ δ[p])
end


add_node1_producer_constraints (generic function with 1 method)

**Producer B (Node 1):**

$0 \leq -\pi_1 + \delta_1^B \perp s_1^B \geq 0$

$0 \leq \gamma_1^B + \lambda_1^B - \delta_1^B \perp q_1^B \geq 0$

$0 \leq -\pi_2 + (\tau_{12}^{Reg} + \tau_{12}) + \delta_1^B \perp f_{12}^B \geq 0$

$0 \leq \bar{q}_1^B - q_1^B \perp \lambda_1^B \geq 0$

$0 = s_1^B - q_1^B + f_{12}^B, \delta_1^B \text{ free}$

Producer B's constraints have the identical structure to producer A's, so we reuse the same `add_node1_producer_constraints` function.


**Producer C (Node 2):**

$0 \leq -\pi_2 + \delta_2^C \perp s_2^C \geq 0$

$0 \leq \gamma_2^C + \lambda_2^C - \delta_2^C \perp q_2^C \geq 0$

$0 \leq \bar{q}_2^C - q_2^C \perp \lambda_2^C \geq 0$

$0 = s_2^C - q_2^C, \delta_2^C \text{ free}$

**Producer D (Node 2):**

$0 \leq -\pi_2 + \delta_2^D \perp s_2^D \geq 0$

$0 \leq \gamma_2^D + \lambda_2^D - \delta_2^D \perp q_2^D \geq 0$

$0 \leq \bar{q}_2^D - q_2^D \perp \lambda_2^D \geq 0$

$0 = s_2^D - q_2^D, \delta_2^D \text{ free}$

Producers C and D at Node 2 have identical structure (no inter-nodal flow term), so we use a single function for both.

In [30]:
function add_node2_producer_constraints(m, p)
    # Pick up variables from the model
    q, s, λ, δ, π_var = m[:q], m[:s], m[:λ], m[:δ], m[:π_var]

    # Producer p (Node 2): s, q, λ complementarity conditions
    @constraint(m, -π_var["node2"] + δ[p] ⟂ s[p])
    @constraint(m, γ[p] + λ[p] - δ[p] ⟂ q[p])
    @constraint(m, q_bar[p] - q[p] ⟂ λ[p])

    # Production balance (equality; δ[p] is the free multiplier)
    @constraint(m, s[p] - q[p] ⟂ δ[p])
end


add_node2_producer_constraints (generic function with 1 method)

**Market Clearing Conditions:**

$0 = [s_1^A + s_1^B] - D_1(\pi_1), \pi_1 \text{ free}$

$0 = [s_2^C + s_2^D + f_{12}^A + f_{12}^B] - D_2(\pi_2), \pi_2 \text{ free}$

In [31]:
function add_market_clearing_conditions(m)
    s, f, π_var = m[:s], m[:f], m[:π_var]
    @constraint(m, s["A"] + s["B"] - (demand_a["node1"] - demand_b["node1"]*π_var["node1"]) ⟂ π_var["node1"])
    @constraint(m, s["C"] + s["D"] + f["A"] + f["B"] - (demand_a["node2"] - demand_b["node2"]*π_var["node2"]) ⟂ π_var["node2"])
end


add_market_clearing_conditions (generic function with 1 method)

**Network Operator's Constraints:**

$0 \leq -\tau_{12}^{Reg} - \tau_{12} + \gamma^{TSO} + \varepsilon_{12} \perp g_{12} \geq 0$

$0 \leq \bar{g}_{12} - g_{12} \perp \varepsilon_{12} \geq 0$

$0 = g_{12} - [f_{12}^A + f_{12}^B], \tau_{12} \text{ free}$

In [32]:
function add_network_operator_constraints(m, gbar)
    g12, ϵ, f, τ = m[:g12], m[:ϵ], m[:f], m[:τ]
    @constraint(m, -τ_reg - τ + γ_tso + ϵ ⟂ g12)
    @constraint(m, gbar - g12 ⟂ ϵ)
    @constraint(m, g12 - (f["A"] + f["B"]) ⟂ τ)
end


add_network_operator_constraints (generic function with 1 method)

Define a function that initializes the model and adds all components

In [33]:
function create_model(gbar)

    # Create the JuMP model, using the PATH solver
    model = Model(PATHSolver.Optimizer)
    set_silent(model)

    # Add variables and constraints
    add_variables(model)
    add_node1_producer_constraints(model, "A")
    add_node1_producer_constraints(model, "B")
    add_node2_producer_constraints(model, "C")
    add_node2_producer_constraints(model, "D")
    add_market_clearing_conditions(model)
    add_network_operator_constraints(model, gbar)

    return model
end


create_model (generic function with 1 method)

**Solve MCP Model**

In [34]:
# Create the model
model = create_model(transmission_params["capacity"])

# Solve the model
optimize!(model)

In [35]:
# Display results
println("=== MULTI-NODE ENERGY NETWORK RESULTS ===")
println("Termination Status: ", termination_status(model))
println("\nProducer Results:")
for p in producers
    node = p in producers_node1 ? "node1" : "node2"
    println("\nProducer ", p, " (Node ", node[end], "):")
    println("  Production (q_", p, "): ", round(value(model[:q][p]), digits=2), " MW")
    println("  Local Sales (s_", p, "): ", round(value(model[:s][p]), digits=2), " MW")
    if p in producers_node1
        println("  Export to Node 2 (f_", p, "): ", round(value(model[:f][p]), digits=2), " MW")
    end
    println("  Capacity Shadow Price (λ_", p, "): ", round(value(model[:λ][p]), digits=2))
end

println("\nMarket Results:")
println("  Node 1 Price (π₁): \$", round(value(model[:π_var]["node1"]), digits=2), "/MWh")
println("  Node 2 Price (π₂): \$", round(value(model[:π_var]["node2"]), digits=2), "/MWh")
println("  Transmission Flow (g₁₂): ", round(value(model[:g12]), digits=2), " MW")
println("  Endogenous Tariff (τ): \$", round(value(model[:τ]), digits=2), "/MWh")
println("  Transmission Capacity Shadow Price (ε): ", round(value(model[:ϵ]), digits=2))


=== MULTI-NODE ENERGY NETWORK RESULTS ===
Termination Status: LOCALLY_SOLVED

Producer Results:

Producer A (Node 1):
  Production (q_A): 10.0 MW
  Local Sales (s_A): 7.44 MW
  Export to Node 2 (f_A): 2.56 MW
  Capacity Shadow Price (λ_A): 2.0

Producer B (Node 1):
  Production (q_B): 3.0 MW
  Local Sales (s_B): 0.56 MW
  Export to Node 2 (f_B): 2.44 MW
  Capacity Shadow Price (λ_B): 0.0

Producer C (Node 2):
  Production (q_C): 5.0 MW
  Local Sales (s_C): 5.0 MW
  Capacity Shadow Price (λ_C): 0.0

Producer D (Node 2):
  Production (q_D): 0.0 MW
  Local Sales (s_D): 0.0 MW
  Capacity Shadow Price (λ_D): 0.0

Market Results:
  Node 1 Price (π₁): $12.0/MWh
  Node 2 Price (π₂): $15.0/MWh
  Transmission Flow (g₁₂): 5.0 MW
  Endogenous Tariff (τ): $2.5/MWh
  Transmission Capacity Shadow Price (ε): 2.0


### Task 2

1. **Parameter Sensitivity**: How does the solution change if we change transmission capacity in the TSO problem (3, 5, 10)? Do some sensitivity analysis.

The following function takes the transmission capacity as input, solves the model, and displays the results.

In [36]:
function solve_and_display_sensitivity_analysis(capacity)

    println("\n", "="^70)
    println("CAPACITY = ", capacity, " MW")
    println("="^70, "\n")

    # Solve
    model = create_model(capacity)
    optimize!(model)

    # Quick display
    println("Status: ", termination_status(model))
    println("π₁ = \$", round(value(model[:π_var]["node1"]), digits=2), "/MWh, π₂ = \$", round(value(model[:π_var]["node2"]), digits=2), "/MWh")
    println("Flow = ", round(value(model[:g12]), digits=2), " MW, Shadow Price (ε) = \$", round(value(model[:ϵ]), digits=2), "/MWh\n")
end


solve_and_display_sensitivity_analysis (generic function with 1 method)

Test different capacities and view results

In [37]:
# Test different capacities - 3, 5 (baseline), 10
for capacity in [3, 5, 10]
    solve_and_display_sensitivity_analysis(capacity)
end


CAPACITY = 3 MW

Status: LOCALLY_SOLVED
π₁ = $12.0/MWh, π₂ = $16.0/MWh
Flow = 3.0 MW, Shadow Price (ε) = $3.0/MWh


CAPACITY = 5 MW

Status: LOCALLY_SOLVED
π₁ = $12.0/MWh, π₂ = $15.0/MWh
Flow = 5.0 MW, Shadow Price (ε) = $2.0/MWh


CAPACITY = 10 MW

Status: LOCALLY_SOLVED
π₁ = $12.0/MWh, π₂ = $15.0/MWh
Flow = 10.0 MW, Shadow Price (ε) = $2.0/MWh

